In [ ]:
# 1. Install Dependencies (if needed)
!pip install requests pandas geopandas "ipython>=8.0" mapboxgl shapely

In [ ]:
# 2. Setup
import requests
from mapboxgl.viz import LinestringViz
from mapboxgl.utils import create_color_stops
from shapely.geometry import shape, GeometryCollection


color_stops = create_color_stops(
    [10, 20, 30, 40, 50],
    colors="RdYlGn"  # red=slow, green=fast
)

# add your Mapbox token here
MAPBOX_TOKEN = ""
BASE_URL = "http://localhost:8000"

def compute_center(features):
    geoms = [shape(f["geometry"]) for f in features]
    collection = GeometryCollection(geoms)
    minx, miny, maxx, maxy = collection.bounds
    return [(minx + maxx)/2, (miny + maxy)/2]

In [ ]:
# 3. Request Aggregated Data
params = {
"day": "Monday",
"period": "AM Peak",
"limit": 10000
}
response = requests.get(f"{BASE_URL}/aggregates/", params=params)
geojson_data = response.json()

In [ ]:
# 4. Visualize in Mapbox
features = [
    {
        "type": "Feature",
        "geometry": f["geometry"],
        "properties": {
            "average_speed": f["average_speed"],
            "road_name": f["road_name"]
        }
    } for f in geojson_data
]
feature_collection = {
    "type": "FeatureCollection",
    "features":features
}


viz = LinestringViz(
    feature_collection,
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=color_stops,
    center=(-81.6557, 30.3322),  # Duval County, FL
    zoom=11,
    line_width_default=2,
    line_stroke="-",
    opacity=0.8
)

viz.show()

In [ ]:
# 5. Request Link Detail Data
params = {
"day": "Monday",
"period": "AM Peak",
}
link_id = 23032463
response = requests.get(f"{BASE_URL}/aggregates/{link_id}", params=params)
link_data = response.json()

In [ ]:
# 6. Visualize Link Detail in Mapbox
feature_collection = {
    "type": "FeatureCollection",
    "features":[{
        "type": "Feature",
        "geometry": link_data["geometry"],
        "properties": {
            "average_speed": link_data["average_speed"],
            "road_name": link_data["road_name"]
        }
    }]
}

viz = LinestringViz(
    feature_collection,
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=color_stops,
    center=compute_center(feature_collection["features"]), #(-81.6557, 30.3322),  # Duval County, FL
    zoom=11,
    line_width_default=2,
    line_stroke="-",
    opacity=0.8
)

viz.show()

In [ ]:
# 7. Request Slow Links
params = {
"period": "AM Peak",
"min_days": 1,
"threshold": 5,
"limit": 10000
}
response = requests.get(f"{BASE_URL}/patterns/slow_links", params=params)
slow_link_data = response.json()

In [ ]:
# 8. Visualize Slow Links in Mapbox
features = [
    {
        "type": "Feature",
        "geometry": f["geometry"],
        "properties": {
            "average_speed": f["average_speed"],
            "road_name": f["road_name"]
        }
    } for f in slow_link_data
]
feature_collection = {
    "type": "FeatureCollection",
    "features":features
}

viz = LinestringViz(
    feature_collection,
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=color_stops,
    center=compute_center(features),
    zoom=11,
    line_width_default=2,
    line_stroke="-",
    opacity=0.8
)

viz.show()

In [ ]:
# 9. Request Links with Spatial Filter
payload = {
    "period": "AM Peak",
    "day": "Monday",
    "limit": 10000,
    "bbox":[-81.705666,30.214278,-81.660004,30.255212]
}
response = requests.post(f"{BASE_URL}/aggregates/spatial_filter", json=payload)
spatial_filter_data = response.json()

In [ ]:
# 10. Visualize Spatial Filter in Mapbox
features = [
    {
        "type": "Feature",
        "geometry": f["geometry"],
        "properties": {
            "average_speed": f["average_speed"],
            "road_name": f["road_name"]
        }
    } for f in spatial_filter_data
]
feature_collection = {
    "type": "FeatureCollection",
    "features":features
}

viz = LinestringViz(
    feature_collection,
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=color_stops,
    center=compute_center(features),
    zoom=11,
    line_width_default=2,
    line_stroke="-",
    opacity=0.8
)

viz.show()

In [ ]:
# 11. Optional: Tabular Summary
import pandas as pd
df = pd.DataFrame([
    {
        "link_id": f["link_id"],
        "avg_speed": f["average_speed"],
        "road_name": f["road_name"],
        "length": f["length"]
    } for f in geojson_data
])
df.sort_values("avg_speed").head(10)